In [2]:
import pandas as pd
import numpy as np
import math
import datetime as dt
from matplotlib import pyplot as plt

import os
import shutil
import glob

from netCDF4 import Dataset
import xarray as xr

from scipy.interpolate import griddata
from scipy.signal import argrelextrema, find_peaks

In [1]:
dir = r'C:\Users\bmaro\OneDrive - University of South Carolina\Research\plume_analysis'
uiuc_directory = dir + r'\data\02_processed\UIUC'
print(uiuc_directory)

C:\Users\bmaro\OneDrive - University of South Carolina\Research\plume_analysis\data\02_processed\UIUC


In [3]:
uiuc_directory = dir + r'\data\02_processed\UIUC'
uiuc_files = [txt for txt in os.listdir(uiuc_directory) if txt.endswith('.csv')]
print(len(uiuc_files))

10035


In [4]:
uiuc_files[0]

'R18B0217.302417.csv'

USC Data

In [5]:
usc_directory = dir + r'\data\02_processed\USC'
usc_files = [txt for txt in os.listdir(usc_directory) if txt.endswith('.txt')]
print(len(usc_files))

9370


In [6]:
uiuc = pd.DataFrame()
uiuc_meta = pd.DataFrame(columns=['file', 'start', 'end', 'elevation', 'longitude', 'latitude', 'zenith', 'azimuth', 'temp_ground', 'pressure'])

for file in uiuc_files:
    f = os.path.join(uiuc_directory, file)
    meta_data = []
    shot_data = []
    
    bin_width = 3.5
    lidar_height = 2
    origin_X = 293946.1   #293946.1075
    origin_Y = 393318.1   #393318.1287
    origin_Z = 233.6      #233.589
    origin_Azimuth = 296
    origin_Zenith = 0
    
    with open(f, 'r') as file:
        content = file.readlines()
        meta = content[0:7]
        header1 = meta[0]
        header2 = meta[1]
        header3 = meta[2]
        header4 = meta[3]
        header5 = meta[4]
        header6 = meta[5]
        header7 = meta[6]
        
        header2 = header2.split(' ')
        campaign = header2[0]
        startDate = f'{header2[1]} {header2[2]}'
        endDate = f'{header2[3]} {header2[4]}'
        elevation = float(header2[5])
        longitude = float(header2[6])
        latitude = float(header2[7])
        zenith = float(header2[8])*-1
        azimuth = float(header2[9])
        temp_ground = float(header2[10])
        pressure = float(header2[11])
        bin_width = float(header5.split()[6])
    
    df = pd.read_csv(f, skiprows=14, nrows=500)
    df['source'] = 'UIUC'
    df['timestamp'] = pd.to_datetime(startDate, dayfirst=True)
    df['distance'] = (df.index+1)*bin_width 
    df['dist_lidar'] = df['distance'] * np.cos(np.radians(zenith)) ### !!! WORKING HERE NEEDS TO BE CORRECTED FOR DISTANCE FROM LIDAR.  
    df['azimuth'] = azimuth
    df['zenith'] = zenith
    df['analog'] = df['0 (mV)']
    df['true_azimuth'] = azimuth+origin_Azimuth
    
    df['x'] = origin_X + df['distance'] * np.cos(np.radians(zenith+origin_Zenith)) * np.sin(np.radians(azimuth+origin_Azimuth))
    df['y'] = origin_Y + df['distance'] * np.cos(np.radians(zenith+origin_Zenith)) * np.cos(np.radians(azimuth+origin_Azimuth))
    df['z'] = origin_Z + df['distance'] * np.sin(np.radians(zenith+origin_Zenith)) + lidar_height

    uiuc = pd.concat([uiuc, df], axis=0, ignore_index=True)

    meta_data.append([f, startDate, endDate, elevation, longitude, latitude, zenith, azimuth, temp_ground, pressure])
    
    # Create DF
    temp_meta_df = pd.DataFrame(meta_data, columns=['file', 'start', 'end', 'elevation', 'longitude', 'latitude', 'zenith', 'azimuth', 'temp_ground', 'pressure'])
    
    # Convert date columns to datetime
    temp_meta_df['start'] = pd.to_datetime(temp_meta_df['start'], errors='coerce', dayfirst=True)
    temp_meta_df['end'] = pd.to_datetime(temp_meta_df['end'], errors='coerce', dayfirst=True)
    uiuc_meta = pd.concat([uiuc_meta, temp_meta_df], ignore_index=True)

uiuc_meta.head()

C:\Users\bmaro\AppData\Local\Temp\ipykernel_30072\2958743569.py:65: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  uiuc_meta = pd.concat([uiuc_meta, temp_meta_df], ignore_index=True)


,file,start,end,elevation,longitude,latitude,zenith,azimuth,temp_ground,pressure
0,C:\Users\bmaro\OneDrive - University of South ...,2018-11-02 17:30:24,2018-11-02 17:30:24,170.0,23.7534,38.0631,20.0,0.0,15.0,1013.0
1,C:\Users\bmaro\OneDrive - University of South ...,2018-11-02 17:30:54,2018-11-02 17:30:54,170.0,23.7534,38.0631,0.2,25.0,15.0,1013.0
2,C:\Users\bmaro\OneDrive - University of South ...,2018-11-02 17:30:58,2018-11-02 17:30:58,170.0,23.7534,38.0631,0.4,25.0,15.0,1013.0
3,C:\Users\bmaro\OneDrive - University of South ...,2018-11-02 17:31:02,2018-11-02 17:31:02,170.0,23.7534,38.0631,0.6,25.0,15.0,1013.0
4,C:\Users\bmaro\OneDrive - University of South ...,2018-11-02 17:31:06,2018-11-02 17:31:06,170.0,23.7534,38.0631,0.8,25.0,15.0,1013.0


In [7]:
usc = pd.DataFrame()
usc_meta = pd.DataFrame(columns=['file', 'start', 'end', 'elevation', 'longitude', 'latitude', 'zenith', 'azimuth', 'temp_ground', 'pressure'])

for file in usc_files:
    f = os.path.join(usc_directory, file)
    meta_data = []
    shot_data = []
    
    bin_width = 7.5
    lidar_height = 2
    origin_X = 293677.5 #293701.1 # 293705.8   #293705.7868 # 293677.5, 393553.7
    origin_Y = 393553.7 #393557.9 #393522.7   #393522.7423
    origin_Z = 337 #236.8 #235.5      #235.4768
    origin_Azimuth = 118 #120 #110
    origin_Zenith = 0
    
    with open(f, 'r') as file:
        content = file.readlines()
    
        # The first seven lines are meta data
        meta = content[0:7]
        
        header1 = meta[0]
        header2 = meta[1]
        header3 = meta[2]
        header4 = meta[3]
        header5 = meta[4]
        header6 = meta[5]
        header7 = meta[6]
        
        header2 = header2.split(' ')
        campaign = header2[0]
        startDate = f'{header2[3]} {header2[4]}'
        endDate = f'{header2[5]} {header2[6]}'
        elevation = float(header2[7])
        longitude = float(header2[8])
        latitude = float(header2[9])
        zenith = float(header2[10])*-1
        azimuth = float(header2[11])
        temp_ground = float(header2[12])
        pressure = float(header2[13])
        bin_width = float(header5.split()[6])
        data = content[7:7+200]
    
    my_data = []
    for row in data:
        d = row.strip().split('\t')
        my_data.append(d)
    df = pd.DataFrame(my_data, columns=['analog', 'photon'])
    df = df.apply(pd.to_numeric)
    df['angle'] = azimuth
    df['start'] = startDate
    df['start'] = pd.to_datetime(df['start'], dayfirst=True)
    df['end'] = endDate
    df['end'] = pd.to_datetime(df['end'], dayfirst=True)
    
    
    df['source'] = 'USC'
    df['timestamp'] = df['start']
    df['distance'] = (df.index+1)*bin_width
    df['dist_lidar'] = df['distance'] * np.cos(np.radians(zenith))
    df['azimuth'] = azimuth
    df['zenith'] = zenith
    # df['analog'] = df['0 (mV)']
    df['true_azimuth'] = azimuth+origin_Azimuth
    
    df['x'] = origin_X + df['distance'] * np.cos(np.radians(zenith+origin_Zenith)) * np.sin(np.radians(azimuth+origin_Azimuth))
    df['y'] = origin_Y + df['distance'] * np.cos(np.radians(zenith+origin_Zenith)) * np.cos(np.radians(azimuth+origin_Azimuth))
    df['z'] = origin_Z + df['distance'] * np.sin(np.radians(zenith+origin_Zenith)) + lidar_height
    df
    usc = pd.concat([usc, df], axis=0, ignore_index=True)

    meta_data.append([f, startDate, endDate, elevation, longitude, latitude, zenith, azimuth, temp_ground, pressure])
    
    # Create DF
    temp_meta_df = pd.DataFrame(meta_data, columns=['file', 'start', 'end', 'elevation', 'longitude', 'latitude', 'zenith', 'azimuth', 'temp_ground', 'pressure'])
    
    # Convert date columns to datetime
    temp_meta_df['start'] = pd.to_datetime(temp_meta_df['start'], errors='coerce', dayfirst=True)
    temp_meta_df['end'] = pd.to_datetime(temp_meta_df['end'], errors='coerce', dayfirst=True)
    usc_meta = pd.concat([usc_meta, temp_meta_df], ignore_index=True)




C:\Users\bmaro\AppData\Local\Temp\ipykernel_30072\675330942.py:81: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  usc_meta = pd.concat([usc_meta, temp_meta_df], ignore_index=True)


In [8]:
uiuc_data = uiuc[['source', 'timestamp', 'distance', 'dist_lidar', 'azimuth', 'true_azimuth', 'zenith', 'analog', 'x','y', 'z']]
usc_data = usc[['source', 'timestamp', 'distance', 'dist_lidar', 'azimuth', 'true_azimuth', 'zenith', 'analog', 'x','y', 'z']]

In [9]:
usc_meta

,file,start,end,elevation,longitude,latitude,zenith,azimuth,temp_ground,pressure
0,C:\Users\bmaro\OneDrive - University of South ...,2018-11-02 11:13:08,2018-11-02 11:13:10,214.0,40.2,88.4,-0.0,-10.0,6.0,1013.0
1,C:\Users\bmaro\OneDrive - University of South ...,2018-11-02 11:13:14,2018-11-02 11:13:16,214.0,40.2,88.4,-0.0,-9.0,6.0,1013.0
2,C:\Users\bmaro\OneDrive - University of South ...,2018-11-02 11:13:49,2018-11-02 11:13:51,214.0,40.2,88.4,-0.0,-10.0,6.0,1013.0
3,C:\Users\bmaro\OneDrive - University of South ...,2018-11-02 11:13:55,2018-11-02 11:13:57,214.0,40.2,88.4,-0.0,-9.0,6.0,1013.0
4,C:\Users\bmaro\OneDrive - University of South ...,2018-11-02 11:14:02,2018-11-02 11:14:03,214.0,40.2,88.4,-0.0,-8.0,6.0,1013.0
...,...,...,...,...,...,...,...,...,...,...
9365,C:\Users\bmaro\OneDrive - University of South ...,2018-11-02 23:47:58,2018-11-02 23:47:58,214.0,40.2,88.4,2.0,3.2,6.0,1013.0
9366,C:\Users\bmaro\OneDrive - University of South ...,2018-11-02 23:48:06,2018-11-02 23:48:06,214.0,40.2,88.4,2.2,3.2,6.0,1013.0
9367,C:\Users\bmaro\OneDrive - University of South ...,2018-11-02 23:48:10,2018-11-02 23:48:10,214.0,40.2,88.4,2.3,3.2,6.0,1013.0
9368,C:\Users\bmaro\OneDrive - University of South ...,2018-11-02 23:48:14,2018-11-02 23:48:14,214.0,40.2,88.4,2.4,3.2,6.0,1013.0


In [10]:
usc_data.describe()

,timestamp,distance,dist_lidar,azimuth,true_azimuth,zenith,analog,x,y,z
count,1874000,1.874000e+06,1.874000e+06,1.874000e+06,1.874000e+06,1.874000e+06,1.874000e+06,1.874000e+06,1.874000e+06,1.874000e+06
mean,2018-11-02 17:35:59.711419392,7.537500e+02,7.528453e+02,2.596254e+00,1.205963e+02,2.349573e+00,1.048354e+01,2.943252e+05,3.931707e+05,3.698896e+02
min,2018-11-02 11:13:08,7.500000e+00,7.469133e+00,-1.000000e+01,1.080000e+02,-3.000000e-01,0.000000e+00,2.936825e+05,3.924390e+05,3.311461e+02
25%,2018-11-02 14:37:19,3.806250e+02,3.794443e+02,2.800000e+00,1.208000e+02,1.000000e+00,3.448500e+00,2.940041e+05,3.929821e+05,3.460685e+02
50%,2018-11-02 17:48:27,7.537500e+02,7.521912e+02,3.000000e+00,1.210000e+02,2.250000e+00,4.716000e+00,2.943254e+05,3.931720e+05,3.609899e+02
75%,2018-11-02 20:27:35,1.126875e+03,1.125710e+03,3.200000e+00,1.212000e+02,3.670000e+00,6.320800e+00,2.946471e+05,3.933618e+05,3.866088e+02
max,2018-11-02 23:48:18,1.500000e+03,1.500000e+03,2.000000e+01,1.380000e+02,5.200000e+00,5.998535e+02,2.951041e+05,3.935514e+05,4.749489e+02
std,NaN,4.330074e+02,4.324889e+02,1.878858e+00,1.878858e+00,1.537333e+00,2.548666e+01,3.723592e+02,2.213669e+02,2.928626e+01


In [11]:
uiuc_data.describe()

,timestamp,distance,dist_lidar,azimuth,true_azimuth,zenith,analog,x,y,z
count,5017500,5.017500e+06,5.017500e+06,5.017500e+06,5.017500e+06,5.017500e+06,5.017500e+06,5.017500e+06,5.017500e+06,5.017500e+06
mean,2018-11-03 00:27:58.551769088,9.393750e+02,9.319593e+02,6.886687e+01,3.648669e+02,5.259432e+00,2.647018e+01,2.940018e+05,3.939791e+05,3.201679e+02
min,2018-11-02 17:30:24,3.750000e+00,2.296213e-16,0.000000e+00,2.960000e+02,-0.000000e+00,0.000000e+00,2.923625e+05,3.929603e+05,2.356000e+02
25%,2018-11-02 21:08:30,4.715625e+02,4.645213e+02,2.400000e+01,3.200000e+02,2.400000e+00,9.811600e+00,2.933547e+05,3.936396e+05,2.574005e+02
50%,2018-11-03 00:34:55,9.393750e+02,9.320898e+02,1.120000e+02,4.080000e+02,5.000000e+00,1.016050e+01,2.939545e+05,3.939773e+05,2.968454e+02
75%,2018-11-03 03:47:52,1.407188e+03,1.399296e+03,1.120000e+02,4.080000e+02,7.600000e+00,1.158210e+01,2.946481e+05,3.943155e+05,3.621417e+02
max,2018-11-03 07:06:23,1.875000e+03,1.875000e+03,1.650000e+02,4.610000e+02,9.000000e+01,1.000000e+03,2.957867e+05,3.947752e+05,2.110600e+03
std,NaN,5.412648e+02,5.399096e+02,4.477829e+01,4.477829e+01,5.323567e+00,8.144805e+01,7.488248e+02,3.990895e+02,9.050350e+01


In [12]:
# time correction
usc_data['timestamp']=usc_data['timestamp'] + pd.Timedelta(hours=7) 

C:\Users\bmaro\AppData\Local\Temp\ipykernel_30072\2273976285.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  usc_data['timestamp']=usc_data['timestamp'] + pd.Timedelta(hours=7)


In [16]:
print(os.getcwd())

c:\Users\bmaro\OneDrive - University of South Carolina\Research\plume_analysis\archive


In [17]:
uiuc_data.to_csv(r'C:\Users\bmaro\OneDrive - University of South Carolina\Research\plume_analysis\data\03_final\uiuc_data.csv') #data\03_final #C:\Users\bmaro\OneDrive - University of South Carolina\Research\plume_analysis\data\03_final

In [18]:
usc_data.to_csv(r'C:\Users\bmaro\OneDrive - University of South Carolina\Research\plume_analysis\data\03_final\usc_data.csv')

In [19]:
usc_data.describe()

,timestamp,distance,dist_lidar,azimuth,true_azimuth,zenith,analog,x,y,z
count,1874000,1.874000e+06,1.874000e+06,1.874000e+06,1.874000e+06,1.874000e+06,1.874000e+06,1.874000e+06,1.874000e+06,1.874000e+06
mean,2018-11-03 00:35:59.711418880,7.537500e+02,7.528453e+02,2.596254e+00,1.205963e+02,2.349573e+00,1.048354e+01,2.943252e+05,3.931707e+05,3.698896e+02
min,2018-11-02 18:13:08,7.500000e+00,7.469133e+00,-1.000000e+01,1.080000e+02,-3.000000e-01,0.000000e+00,2.936825e+05,3.924390e+05,3.311461e+02
25%,2018-11-02 21:37:19,3.806250e+02,3.794443e+02,2.800000e+00,1.208000e+02,1.000000e+00,3.448500e+00,2.940041e+05,3.929821e+05,3.460685e+02
50%,2018-11-03 00:48:27,7.537500e+02,7.521912e+02,3.000000e+00,1.210000e+02,2.250000e+00,4.716000e+00,2.943254e+05,3.931720e+05,3.609899e+02
75%,2018-11-03 03:27:35,1.126875e+03,1.125710e+03,3.200000e+00,1.212000e+02,3.670000e+00,6.320800e+00,2.946471e+05,3.933618e+05,3.866088e+02
max,2018-11-03 06:48:18,1.500000e+03,1.500000e+03,2.000000e+01,1.380000e+02,5.200000e+00,5.998535e+02,2.951041e+05,3.935514e+05,4.749489e+02
std,NaN,4.330074e+02,4.324889e+02,1.878858e+00,1.878858e+00,1.537333e+00,2.548666e+01,3.723592e+02,2.213669e+02,2.928626e+01


In [21]:
usc_meta.to_csv(r'C:\Users\bmaro\OneDrive - University of South Carolina\Research\plume_analysis\data\03_final\usc_meta.csv')
uiuc_meta.to_csv(r'C:\Users\bmaro\OneDrive - University of South Carolina\Research\plume_analysis\data\03_final\uiuc_meta.csv')

In [ ]:
meta_df = pd.DataFrame(columns=['file', 'start', 'end', 'elevation', 'longitude', 'latitude', 'zenith', 'azimuth', 'temp_ground', 'pressure'])

# Convert appropriate columns to numeric
numeric_cols = ['elevation', 'longitude', 'latitude', 'zenith', 'azimuth', 'temp_ground', 'pressure']
# meta_df[numeric_cols] = meta_df[numeric_cols].apply(pd.to_numeric, errors='coerce')  # Convert and set non-numeric values to NaN

for txt_file in files_to_read:
    f = os.path.join(directory, txt_file)
    meta_data = []
    shot_data = []
    with open(f, 'r') as file:
        content = file.readlines()
    
        # The first seven lines are meta data
        meta = content[0:7]
        
        header1 = meta[0]
        header2 = meta[1]
        header3 = meta[2]
        header4 = meta[3]
        header5 = meta[4]
        header6 = meta[5]
        header7 = meta[6]
        
        header2 = header2.split(' ')
        campaign = header2[0]
        startDate = f'{header2[3]} {header2[4]}'
        endDate = f'{header2[5]} {header2[6]}'
        elevation = float(header2[7])
        longitude = float(header2[8])
        latitude = float(header2[9])
        zenith = float(header2[10])*-1
        azimuth = float(header2[11])
        temp_ground = float(header2[12])
        pressure = float(header2[13])
        bin_width = float(header5.split()[6])
        data = content[7:]
    
    # Create a dataframe for the meta data
    meta_data.append([f, startDate, endDate, elevation, longitude, latitude, zenith, azimuth, temp_ground, pressure])
    
    # Create DF
    temp_meta_df = pd.DataFrame(meta_data, columns=['file', 'start', 'end', 'elevation', 'longitude', 'latitude', 'zenith', 'azimuth', 'temp_ground', 'pressure'])
    
    # Convert date columns to datetime
    temp_meta_df['start'] = pd.to_datetime(temp_meta_df['start'], errors='coerce', dayfirst=True)
    temp_meta_df['end'] = pd.to_datetime(temp_meta_df['end'], errors='coerce', dayfirst=True)
    meta_df = pd.concat([meta_df, temp_meta_df], ignore_index=True)
meta_df = meta_df.sort_values('start')
print( meta_df)

NameError: name 'files_to_read' is not defined